<a href="https://colab.research.google.com/github/ricardoalmeida/ai-post-generator/blob/main/Google_ADK_AI_Agent_for_social_media.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.

In [3]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [4]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

In [5]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)

In [6]:
# Cria um serviço de sessão em memória

session_service = InMemorySessionService()

In [7]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final

async def call_agent(agent: Agent, message_text: str) -> str:
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session = await session_service.create_session(app_name=agent.name, user_id="user1")
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    async for event in runner.run_async(user_id="user1", session_id=session.id, new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [8]:
# Lista todos os modelos disponíveis atualmente

for model in client.models.list():
    print(model.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-pro-preview-05-06
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview

In [9]:
MODELO_RAPIDO = "gemini-2.0-flash"
MODELO_ROBUSTO = "gemini-2.5-pro-preview-03-25"

In [10]:
##########################################
# --- Agente 1: Buscador de Notícias --- #
##########################################

async def agente_buscador(topico, data_de_hoje):

    buscador = Agent(
        name="agente_buscador",
        model=MODELO_RAPIDO,
        instruction="""
        Você é um assistente de pesquisa. A sua tarefa é usar a ferramenta de busca do google (google_search)
        para recuperar as últimas notícias de lançamentos muito relevantes sobre o tópico abaixo.
        Foque em no máximo 5 lançamentos relevantes, com base na quantidade e entusiasmo das notícias sobre ele.
        Se um tema tiver poucas notícias ou reações entusiasmadas, é possível que ele não seja tão relevante assim
        e pode ser substituído por outro que tenha mais.
        Esses lançamentos relevantes devem ser atuais, de no máximo um mês antes da data de hoje.
        """,
        description="Agente que busca informações no Google",
        tools=[google_search]
    )

    entrada_do_agente_buscador = f"Tópico: {topico}\nData de hoje: {data_de_hoje}"

    # Executa o agente
    lancamentos = await call_agent(buscador, entrada_do_agente_buscador)
    return lancamentos

In [11]:
################################################
# --- Agente 2: Planejador de posts --- #
################################################

async def agente_planejador(topico, lancamentos_buscados):
    planejador = Agent(
        name="agente_planejador",
        model=MODELO_RAPIDO,
        instruction="""
        Você é um planejador de conteúdo, especialista em redes sociais.
        Você recebe uma lista de lançamentos recentes do agente buscador e,
        - Para cada um dos lançamentos recebidos, você deve usar a ferramenta
        de busca do Google (google_search) para buscar os pontos mais relevantes
        que poderíamos abordar em um post sobre cada um deles.
        Você também pode usar o (google_search) para encontrar mais
        informações sobre cada um dos temas e aprofundar.
        - Depois de terminar a busca, você irá escolher APENAS UM tema dentre todos eles,
        aquele tema com potencial de ser o mais relevante com base nas suas pesquisas.
        - Depois de escolher o tema mais relevante dentre todos eles, você irá retornar
        qual foi o tema escolhido, seus pontos mais relevantes, e um plano com os assuntos
        a serem abordados no post que será escrito posteriormente.
        """,
        description="Agente que planeja posts",
        tools=[google_search]
    )

    entrada_do_agente_planejador = f"Tópico:{topico}\nLançamentos buscados: {lancamentos_buscados}"

    # Executa o agente
    plano_do_post = await call_agent(planejador, entrada_do_agente_planejador)
    return plano_do_post

In [12]:
######################################
# --- Agente 3: Redator do Post --- #
######################################

async def agente_redator(topico, plano_de_post):
    redator = Agent(
        name="agente_redator",
        model=MODELO_ROBUSTO,
        instruction="""
            Você é um Redator Criativo especializado em criar posts virais para redes sociais.
            Você escreve posts para a empresa Alura, a maior escola online de tecnologia do Brasil.
            Utilize o tema fornecido no plano de post e os pontos mais relevantes fornecidos e, com base nisso,
            escreva um post para Instagram sobre o tema indicado.
            O post deve ser engajador, informativo, com linguagem simples e incluir 2 a 4 hashtags no final.
            """,
        description="Agente redator de posts engajadores para Instagram"
    )
    entrada_do_agente_redator = f"Tópico: {topico}\nPlano de post: {plano_de_post}"

    # Executa o agente
    post_final = await call_agent(redator, entrada_do_agente_redator)
    return post_final

In [13]:
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab

# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [14]:
from datetime import date

data_de_hoje = date.today().strftime("%d/%m/%Y")

In [15]:
print("🚀 Iniciando o Sistema de Criação de Posts para Instagram com 3 Agentes 🚀\n")

# --- Obter o Tópico do Usuário ---
topico = input("❓ Por favor, digite o TÓPICO sobre o qual você quer criar o post de tendências: ")

# Inserir lógica do sistema de agentes
if not topico:
    print("\nVocê esqueceu de digitar o tópico!")
else:
    print(f"\nMaravilha! Vamos então criar o post sobre novidades em {topico}")

    lancamentos_buscados = await agente_buscador(topico, data_de_hoje)
    print("\n--- 📝 Resultado do Agente 1 (Buscador) ---\n")
    display(to_markdown(lancamentos_buscados))
    print("--------------------------------------------------------------")

    plano_de_post = await agente_planejador(topico, lancamentos_buscados)
    print("\n--- 📝 Resultado do Agente 2 (Planejador) ---\n")
    display(to_markdown(plano_de_post))
    print("--------------------------------------------------------------")

    post_redigido = await agente_redator(topico, plano_de_post)
    print("\n--- 📝 Resultado do Agente 3 (Redator) ---\n")
    display(to_markdown(post_redigido))
    print("--------------------------------------------------------------")

🚀 Iniciando o Sistema de Criação de Posts para Instagram com 3 Agentes 🚀

❓ Por favor, digite o TÓPICO sobre o qual você quer criar o post de tendências: Tendências em Inteligência Artificial

Maravilha! Vamos então criar o post sobre novidades em Tendências em Inteligência Artificial

--- 📝 Resultado do Agente 1 (Buscador) ---



> Para encontrar as tendências mais recentes em Inteligência Artificial, vou usar o Google para pesquisar os lançamentos e desenvolvimentos mais comentados no último mês. As buscas serão focadas em identificar novidades que geraram grande interesse e cobertura noticiosa.
> 
> 
> Com base nas minhas buscas, aqui estão algumas das tendências mais relevantes em Inteligência Artificial em maio de 2025:
> 
> 1.  **Google I/O 2025 e Avanços do Gemini:** O Google I/O 2025 revelou grandes avanços, principalmente nos modelos Gemini 2.5 Pro e Flash, que oferecem raciocínio avançado, geração de código precisa e respostas mais rápidas com maior eficiência. Além disso, o Google apresentou o "AI Mode" no Google Search, "Deep Search" e pesquisa multimodal, que representam uma grande transformação na forma como interagimos com a busca. O Gemini se tornou um assistente universal, integrando-se a vários aplicativos e plataformas.
> 2.  **Agentes de IA Autônomos:** Agentes de IA capazes de automatizar e otimizar processos de forma autônoma estão se tornando uma tendência importante. Projetos como AgentGPT da OpenAI e Gemini 2.0 do Google visam o desenvolvimento de agentes que podem realizar tarefas complexas de forma independente, como responder e-mails e gerenciar fluxos de trabalho. A Microsoft também apresentou novidades em agentes de IA para empresas.
> 3.  **IA Generativa e Criação de Conteúdo:** A IA generativa continua a se expandir, com ferramentas como Imagen 4, Veo 3, Lyria 2 e Flow, transformando a criação de conteúdo digital. A capacidade de gerar vídeos mais longos e de alta qualidade, juntamente com a criação de imagens realistas e tipografia aprimorada, está impactando setores como marketing, educação e entretenimento. Ferramentas como o Google Veo auxiliam cineastas e roteiristas na produção de filmes.
> 4.  **Aplicações de IA na Saúde:** A IA está cada vez mais presente na área da saúde, oferecendo diagnósticos assistidos, análise de imagens médicas e desenvolvimento de medicamentos. Há também um foco crescente em "médicos centauros", que combinam a análise de máquinas com a intuição humana para melhorar as decisões médicas.
> 5.  **Edge AI:** A Azion lançou o Edge AI, que permite que empresas implementem IA mais perto de onde os dados são gerados e consumidos, reduzindo a latência, diminuindo custos e garantindo conformidade com regulamentações de privacidade. Isso possibilita a execução de modelos de IA poderosos em infraestruturas distribuídas.
> 
> Essas tendências refletem um cenário de rápida evolução, onde a IA está se tornando mais acessível, eficiente e integrada em diversas áreas da sociedade e da indústria.


--------------------------------------------------------------

--- 📝 Resultado do Agente 2 (Planejador) ---



> Obrigado pela lista de lançamentos e tendências em IA! Analisando os temas, o que me parece ter o maior potencial para um post de destaque nas redes sociais é o primeiro:
> 
> **Google I/O 2025 e Avanços do Gemini**
> 
> **Por que este tema?**
> 
> *   **Alto interesse público:** Os anúncios do Google I/O sempre geram grande expectativa e cobertura midiática, atraindo um público amplo interessado em tecnologia e inovação.
> *   **Transformação na busca:** O "AI Mode" e o "Deep Search" têm o potencial de mudar fundamentalmente como as pessoas encontram informações online, despertando curiosidade e debates.
> *   **Aplicações práticas:** A integração do Gemini em diversos aplicativos e plataformas demonstra o impacto direto da IA no dia a dia dos usuários, tornando o tema mais relevante e tangível.
> 
> **Plano de Conteúdo para o Post**
> 
> 1.  **Título:** "Google I/O 2025: Gemini Revoluciona a Busca e Integra-se ao Seu Mundo!" (ou similar, chamativo e direto ao ponto)
> 2.  **Introdução:**
>     *   Começar com uma pergunta impactante: "Já imaginou uma busca na internet que realmente 'entende' o que você precisa?"
>     *   Breve resumo do Google I/O 2025 e a importância dos anúncios relacionados à IA.
>     *   Destacar que o Gemini está se tornando um "assistente universal".
> 3.  **Tópicos a serem abordados:**
>     *   **Gemini 2.5 Pro e Flash:**
>         *   Explicar as melhorias em raciocínio, geração de código e velocidade.
>         *   Usar exemplos práticos de como essas melhorias beneficiam os usuários.
>     *   **"AI Mode" e "Deep Search" no Google Search:**
>         *   Descrever como essas novas ferramentas transformam a experiência de busca.
>         *   Mostrar exemplos de pesquisas complexas que podem ser resolvidas com o "Deep Search".
>         *   Enfatizar como a busca multimodal permite combinar diferentes tipos de informação (texto, imagem, vídeo).
>     *   **Integração do Gemini em aplicativos e plataformas:**
>         *   Detalhar como o Gemini está sendo incorporado em ferramentas como Gmail, Docs, etc.
>         *   Mostrar exemplos de como o Gemini pode ajudar os usuários a serem mais produtivos e criativos.
> 4.  **Visuals:**
>     *   Usar imagens e vídeos do Google I/O 2025 para ilustrar os anúncios.
>     *   Criar GIFs ou vídeos curtos mostrando exemplos práticos do "AI Mode" e do "Deep Search".
> 5.  **Chamada para Ação:**
>     *   Convidar os seguidores a experimentar as novas funcionalidades do Google.
>     *   Incentivar a discussão nos comentários: "O que você espera do futuro da busca com IA?"
> 6.  **Hashtags:**
>     *   #GoogleIO2025 #Gemini #IA #InteligenciaArtificial #Busca #Tecnologia #Inovacao #AIMode #DeepSearch
> 
> **Pesquisas Adicionais:**
> 
> Para enriquecer o post, sugiro pesquisar mais sobre:
> 
> 
> Excelente! Com base nas pesquisas adicionais, aqui estão alguns pontos adicionais para enriquecer o post sobre o Google I/O 2025 e o Gemini:
> 
> *   **Gemini Ultra:** Detalhes sobre o plano de assinatura premium, incluindo o acesso ao Veo 3, Gemini 2.5 Pro com modo Deep Think, Flow, Project Mariner e YouTube Premium. Mencionar o preço (US$249,99/mês) e o espaço de armazenamento na nuvem (30TB).
> *   **Veo 3:** Exemplos de como o Veo 3 auxilia cineastas e roteiristas na produção de filmes, com vídeos mais longos e de alta qualidade.
> *   **Project Mariner:** Explicar como o Project Mariner permite a navegação e interação autônoma com websites para realizar tarefas como compras online sem intervenção direta do usuário.
> *   **AI Mode:** Explicar como o AI Mode no Google Search representa uma "reimaginação total da busca" e como ele oferece respostas mais detalhadas e personalizadas.
> *   **Deep Search:** Detalhes sobre como o Deep Search utiliza visualização e análise de dados, recomendações de compras baseadas em IA, ações de compras assistidas por agentes e personalização com o Gemini.
> *   **Testes com Gemini 2.5:** O Google está testando vários recursos novos no AI Mode, incluindo Deep Search, Live Search, contexto pessoal, gráficos personalizados, compras com IA e recursos de agente.
> *   **Impacto no SEO:** O AI Mode pode reduzir cliques para sites tradicionais, reforçando a necessidade de criar conteúdos estruturados, originais e bem direcionados para serem aproveitados dentro das respostas da IA. Otimização para Generative Engine Optimisation (GEO).
> 
> Com essas informações adicionais, o post terá ainda mais valor para o público e poderá gerar grande engajamento nas redes sociais.


--------------------------------------------------------------


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': "Gemini 2.5 Pro Preview doesn't have a free quota tier. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.", 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_input_token_count', 'quotaId': 'GenerateContentInputTokensPerModelPerDay-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-pro-exp'}}, {'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-pro-exp'}}, {'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'glob